# Week 10. 머신러닝 모델과 웹서비스 흐름

10주차는 모델 학습, 직렬화, 예측 API, UI 연결 흐름을 다룹니다.
실제 서버를 띄우지 않고도 모델 학습과 예측 응답 생성까지 한 노트북에서 검증합니다.


## 제출자 정보

- 이름: 이성민
- 학과: 소프트웨어융합과
- 학년: 2학년
- 학번: 2151050


## 1. Iris 데이터로 모델 학습하기

scikit-learn의 내장 데이터셋을 사용하면 외부 파일 없이도 머신러닝 흐름을 실행할 수 있습니다.


In [1]:
import pickle

import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data,
    iris.target,
    test_size=0.25,
    random_state=2026,
    stratify=iris.target,
)

model = RandomForestClassifier(n_estimators=50, random_state=2026)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print(f"테스트 정확도: {accuracy_score(y_test, predictions):.3f}")


테스트 정확도: 0.921


## 2. 모델 직렬화와 로드 확인

웹서비스에서는 학습과 예측을 분리하는 경우가 많습니다.
여기서는 파일을 남기지 않고 바이트로 직렬화해 같은 개념을 확인합니다.


In [2]:
model_bytes = pickle.dumps(model)
loaded_model = pickle.loads(model_bytes)

sample = np.array([[5.1, 3.5, 1.4, 0.2]])
sample_prediction = loaded_model.predict(sample)[0]
print("샘플 예측:", iris.target_names[sample_prediction])


샘플 예측: setosa


## 3. 예측 API 응답 함수 만들기

FastAPI의 엔드포인트는 요청 모델을 받고 JSON으로 응답합니다.
여기서는 Pydantic 요청 모델과 일반 함수로 같은 구조를 재현합니다.


In [3]:
from pydantic import BaseModel, Field


class IrisRequest(BaseModel):
    sepal_length: float = Field(gt=0)
    sepal_width: float = Field(gt=0)
    petal_length: float = Field(gt=0)
    petal_width: float = Field(gt=0)


def predict_species(request: IrisRequest):
    data = [[
        request.sepal_length,
        request.sepal_width,
        request.petal_length,
        request.petal_width,
    ]]
    predicted_class = int(loaded_model.predict(data)[0])
    probability = loaded_model.predict_proba(data)[0]
    return {
        "species": str(iris.target_names[predicted_class]),
        "confidence": round(float(probability.max()), 4),
    }


predict_species(IrisRequest(sepal_length=6.3, sepal_width=3.3, petal_length=6.0, petal_width=2.5))


{'species': 'virginica', 'confidence': 1.0}

## 4. UI 함수 형태로 감싸기

Gradio UI는 결국 사용자가 넣은 숫자를 Python 함수에 전달하고, 함수의 반환값을 화면에 보여줍니다.
그래서 UI를 붙이기 전에도 함수만으로 동작을 검증할 수 있습니다.


In [4]:
def ui_predict(sepal_length, sepal_width, petal_length, petal_width):
    result = predict_species(
        IrisRequest(
            sepal_length=sepal_length,
            sepal_width=sepal_width,
            petal_length=petal_length,
            petal_width=petal_width,
        )
    )
    return f"{result['species']} ({result['confidence'] * 100:.1f}%)"


ui_predict(5.9, 3.0, 5.1, 1.8)


'virginica (80.0%)'

## 정리

- 모델 학습, 저장, 로드, 예측은 웹서비스 배포의 기본 단계입니다.
- 요청 모델은 입력값을 안전하게 검증합니다.
- API와 UI는 같은 예측 함수를 공유하면 중복을 줄일 수 있습니다.
